# TaqSim block by block

## Seven kinds of block, one small river, and the rule card each block carries

Dr. Tobias Siegfried, hydrosolutions GmbH · CC BY 4.0

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

In [ ]:
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
# A runtime that ran this cell earlier may hold an older copy of the course: bring the folder up to date and
# forget any course module Python has already loaded, so that the current code is what gets imported.
if os.path.exists(".git"):
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
for name in [m for m in sys.modules if m == "zarafshan_taqsim" or m.startswith("zarafshan_taqsim.")]:
    del sys.modules[name]
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
#| label: fig-catalogue
#| fig-cap: "The seven kinds of block. Every TaqSim model, including the 61-block Zarafshan model of this course, is made of these and nothing else."
#| echo: false
from zarafshan_taqsim.blocks_demo import plot_block_catalogue

_ = plot_block_catalogue()

Three things are true of every block:

- **A block only moves water.** It receives, keeps, loses, hands on. It never looks at the rest of the network.
- **A block's rule is a card, not a brain.** The rule is a few lines of Python: *given what arrived this month and
  what I hold, do this*. You can read every rule in this notebook in under a minute.
- **A block keeps a diary.** Every month it writes down what it received, stored, lost, released or failed to
  deliver. These diary entries are called *events*, and reading a result means reading the diaries.

Building any TaqSim model is always the same four steps: create an empty system, add the blocks, add the edges,
check and run. Time moves in steps; here one step is one month, the year starts in October like the hydrological
year of the Zarafshan, and volumes are in million cubic metres (Mm³).

## Setup {#sec-setup}

In [ ]:
#| label: setup
from dataclasses import dataclass
from typing import ClassVar

import numpy as np
import pandas as pd

# The seven kinds of block, the edge, the system and the pieces a rule is made of
from taqsim import Demand, Edge, Frequency, PassThrough, Reach, Sink, Source, Splitter, Storage, Strategy, TimeSeries, Timestep, WaterSystem
from taqsim.node import EVAPORATION, SEEPAGE, NoReachLoss, NoRouting

# The diary entries we will read
from taqsim.node import DeficitRecorded, WaterConsumed, WaterGenerated, WaterLost, WaterOutput, WaterPassedThrough, WaterReceived, WaterReleased, WaterSpilled, WaterStored

# This notebook's helpers (src/zarafshan_taqsim/blocks_demo.py): the numbers, the rules, the figures
from zarafshan_taqsim.blocks_demo import (
    CAPACITY_MM3,
    DEAD_STORAGE_MM3,
    FARM_NEED_MM3,
    INFLOW_MM3,
    MONTHS,
    START_STORAGE_MM3,
    CanalShare,
    Evaporation,
    Knobs,
    LagRouting,
    ReleaseTheNeed,
    SeepageLoss,
    balance_table,
    build_demo,
    monthly,
    plot_balance,
    plot_demo_network,
    plot_months,
    plot_where_the_water_went,
    rule_card,
    simulate_demo,
    system_balance,
)

river = np.array(INFLOW_MM3)  # the Zarafshan at Ravatkhoja, median month of 2010-2023, October first
print(f"The river brings {river.sum():,.0f} Mm³ a year: {river.min():.0f} in its lowest month, {river.max():.0f} in its highest.")

# Source and Sink: the spring and the sea {#sec-source-sink}

The two ends of every line. A **Source** puts water into the network each month, from a list of numbers you give
it. A **Sink** takes water out and records it. Neither has a rule to write: the Source's "rule" is its list, and the
Sink has nothing to decide.

The smallest system is one Source, one Sink and one edge. Here are the four steps, in full.

In [ ]:
#| label: source-sink
system = WaterSystem(frequency=Frequency.MONTHLY)  # 1. an empty system; one step is one month
system.add_node(Source(id="river", inflow=TimeSeries(values=list(river))))  # 2. the blocks
system.add_node(Sink(id="sea"))
system.add_edge(Edge(id="river_to_sea", source="river", target="sea"))  # 3. the edge, named after its two ends
system.validate()  # 4. check that the network is complete, then run twelve months
system.simulate(12)

generated = monthly(system.nodes["river"], WaterGenerated, steps=12)  # the river's diary: what it put in
arrived = monthly(system.nodes["sea"], WaterReceived, steps=12)  # the sea's diary: what reached it
print("Every month the sea received exactly what the river generated:", np.allclose(generated, arrived))

In [ ]:
#| label: fig-source
#| fig-cap: "The Source puts in what its list says: the Zarafshan's median year, low in winter, a melt peak in July."
#| echo: false
_ = plot_months({"the river": generated}, "A Source: one number per month, from a list")

Two small things worth knowing already. The edge's name `river_to_sea` appears in the sea's diary as the
`source_id` of each delivery, which is how a block with several inflows tells them apart. And `monthly(...)` is a
four-line helper that sums one kind of diary entry per month; you will see it under every figure.

# Reach: a corridor {#sec-reach}

A **Reach** is a stretch of river or canal. Water that enters it comes out later, and some of it leaks on the way.
It carries two cards: a *routing model* that says how long water takes, and a *loss rule* that says how much leaks.
Our routing model holds each month's water for one month; our loss rule lets a tenth of the flow seep into the
ground.

In [ ]:
#| label: reach-cards
#| echo: false
rule_card(LagRouting)

In [ ]:
#| label: reach-loss-card
#| echo: false
rule_card(SeepageLoss)

Read the routing card from the top: `initial_state` is what the corridor holds before the first month (nothing);
`route` is called every month with what entered, returns what comes out now and what the corridor keeps; `storage`
says how much is inside. The loss card returns a dictionary of losses by reason, so that the diary can say *why*
water was lost. Neither card is a `Strategy`: physics is not something the optimizer should tune.

In [ ]:
#| label: reach
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="river", inflow=TimeSeries(values=list(river))))
system.add_node(Reach(id="canyon", routing_model=LagRouting(months=1), loss_rule=SeepageLoss(share=0.1)))
system.add_node(Sink(id="sea"))
system.add_edge(Edge(id="river_to_canyon", source="river", target="canyon"))
system.add_edge(Edge(id="canyon_to_sea", source="canyon", target="sea"))
system.validate()
system.simulate(12)

canyon = system.nodes["canyon"]
entered = monthly(canyon, WaterReceived, steps=12)
leaked = monthly(canyon, WaterLost, steps=12)
came_out = monthly(canyon, WaterOutput, steps=12)

In [ ]:
#| label: fig-reach
#| fig-cap: "What enters the canyon comes out one month later and a tenth smaller. In October nothing comes out yet; the first month's water is still inside."
#| echo: false
_ = plot_months({"entered the canyon": entered, "came out": came_out, "leaked away": leaked}, "A Reach: delay and loss")

The July peak of 993 Mm³ reaches the far end in August, as 894. Over the year the corridor kept back the last
month's water, so the sea received less than the river gave; the difference is sitting in the canyon. The
Zarafshan model of this course has 35 reaches, one for every stretch of river and canal between two other blocks.

# Splitter: a fork with a traffic officer {#sec-splitter}

A **Splitter** sends water to two or more places. Its card, the *split rule*, must say how much goes to each
target, by the target's name. Ours sends a fixed share into the canal and the rest on down the river.

In [ ]:
#| label: split-card
#| echo: false
rule_card(CanalShare)

This card *is* a `Strategy`: `__params__` names the knob the optimizer may turn (`share`), and `__bounds__` says
between which values. The dictionary it returns is keyed by the names of the targets, so a split rule and the edges
of its block must agree on those names.

In [ ]:
#| label: splitter
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="river", inflow=TimeSeries(values=list(river))))
system.add_node(Splitter(id="headworks", split_policy=CanalShare(share=0.3)))
system.add_node(Sink(id="canal"))  # for now the two branches end right here
system.add_node(Sink(id="turbine"))
for a, b in (("river", "headworks"), ("headworks", "canal"), ("headworks", "turbine")):
    system.add_edge(Edge(id=f"{a}_to_{b}", source=a, target=b))
system.validate()
system.simulate(12)

to_canal = monthly(system.nodes["canal"], WaterReceived, steps=12)
to_river = monthly(system.nodes["turbine"], WaterReceived, steps=12)

In [ ]:
#| label: fig-splitter
#| fig-cap: "Thirty percent into the canal, seventy down the river, every month. The two bars always add up to the river."
#| echo: false
_ = plot_months({"into the canal": to_canal, "down the river": to_river}, "A Splitter: so much this way, the rest that way", kind="bar", stacked=True)

A split rule may look at the month (`t.index`), at the amount, or at anything the block knows. A rule that sends
more in summer than in winter is three lines longer; exercise 1 asks you to write it.

# Storage: a bathtub with a valve {#sec-storage}

A **Storage** block is a reservoir. Picture a bathtub: water pours in, the tub fills, when it is full the rest
overflows, some goes up as vapour, and the valve at the bottom lets out what you tell it to. Below the valve there is
water the valve cannot reach (the *dead storage*). The tub carries two cards: a *release rule* for the valve and a
*loss rule* for the vapour.

In [ ]:
#| label: release-card
#| echo: false
rule_card(ReleaseTheNeed)

In [ ]:
#| label: evaporation-card
#| echo: false
rule_card(Evaporation)

The release rule reads the farm's need for this month from its list, and opens the valve for that much, but never
more than the valve's maximum (`cap`) and never more than there is above the dead storage. The loss rule takes three
percent of whatever is in the tub.

Every month the tub does things in a fixed order: **receive, store (overflow if full), lose, release**. The order
matters: evaporation is taken from the water that includes this month's inflow, and the valve sees what is left
after that.

In [ ]:
#| label: storage
canal_water = river * 0.2  # the canal's share, as if the headworks sent a fifth of the river into the tub
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="canal", inflow=TimeSeries(values=list(canal_water) * 2)))  # two years: the second is shown
system.add_node(
    Storage(
        id="reservoir",
        capacity=CAPACITY_MM3,  # 667 Mm³, the Kattakurgan reservoir
        dead_storage=DEAD_STORAGE_MM3,  # 66.7 Mm³ the valve cannot reach
        initial_storage=START_STORAGE_MM3,  # 170 Mm³ on the first of October
        release_policy=ReleaseTheNeed(cap=60.0),
        loss_rule=Evaporation(share=0.03),
    )
)
system.add_node(Sink(id="farm"))
system.add_edge(Edge(id="canal_to_reservoir", source="canal", target="reservoir"))
system.add_edge(Edge(id="reservoir_to_farm", source="reservoir", target="farm"))
system.validate()
system.simulate(24)

tub = system.nodes["reservoir"]
second_year = slice(12, 24)
stored = monthly(tub, WaterStored, steps=24)  # what went into the tub each month (not the level)
overflowed = monthly(tub, WaterSpilled, steps=24)
evaporated = monthly(tub, WaterLost, steps=24)
released = monthly(tub, WaterReleased, steps=24)
level = START_STORAGE_MM3 + np.cumsum(stored - evaporated - released)  # the level at the end of each month
stored, overflowed, evaporated, released, level = (x[second_year] for x in (stored, overflowed, evaporated, released, level))
print(f"The tub ends the second year holding {tub.storage:,.0f} Mm³; the level path says {level[-1]:,.0f}.")

In [ ]:
#| label: fig-storage
#| fig-cap: "The second year of the tub. From May to August the melt does not fit and the rest overflows; the valve lets out the farm's need, at most 60; the vapour takes three percent a month, about 19 Mm³. The dashed line is the need."
#| echo: false
_ = plot_months(
    {"went into the tub": stored, "overflowed": overflowed, "evaporated": evaporated, "released by the valve": released},
    "A Storage: receive, store, lose, release",
    kind="bar",
    reference=("the farm's need", np.array(FARM_NEED_MM3)),
)

In [ ]:
#| label: fig-level
#| fig-cap: "The level at the end of each month of the second year, against the rim (667) and the dead storage (66.7). The level never shows 667 at month end: within the month the valve and the vapour already took their share."
#| echo: false
fig = plot_months({"water in the tub": level}, "How full the tub is", unit="Mm³")
ax = fig.axes[0]
ax.axhline(CAPACITY_MM3, color="#6b6a66", lw=1, ls="--")
ax.axhline(DEAD_STORAGE_MM3, color="#6b6a66", lw=1, ls=":")
ax.set_ylim(0, 720)

One thing to remember from the overflow: in TaqSim, **a full reservoir's overflow travels on along the outgoing
edge**, here to the farm, exactly like a release. It is recorded as `WaterSpilled` so that you can tell the two
apart. We will meet a different kind of spill at the gate.

# Demand: a customer {#sec-demand}

A **Demand** is a user of water: a farm, a city, a power plant. Its card has three numbers. The *requirement* is
what it asks for each month, from a list. The *consumption fraction* is the share of what it gets that it uses up;
the rest goes back downstream, as drainage or cooling water. The *efficiency* is the share of the water at its gate
that reaches the point of use; the rest is lost on the way, between the canal and the roots.

When the customer gets less than it asked for, it writes a `DeficitRecorded` entry with the shortfall. That entry
is the one every score in this course is built from.

In [ ]:
#| label: demand
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="canal", inflow=TimeSeries(values=list(canal_water))))
system.add_node(
    Demand(
        id="farm",
        requirement=TimeSeries(values=list(FARM_NEED_MM3)),  # what the crops need each month
        consumption_fraction=0.7,  # the crops use up 70 %; 30 % drains back
        efficiency=0.85,  # 85 % of the water at the gate reaches the roots
    )
)
system.add_node(Sink(id="drain"))
system.add_edge(Edge(id="canal_to_farm", source="canal", target="farm"))
system.add_edge(Edge(id="farm_to_drain", source="farm", target="drain"))
system.validate()
system.simulate(12)

farm = system.nodes["farm"]
received = monthly(farm, WaterReceived, steps=12)
lost_on_the_way = monthly(farm, WaterLost, steps=12)
used_up = monthly(farm, WaterConsumed, steps=12)
drained_back = monthly(farm, WaterOutput, steps=12)
short = monthly(farm, DeficitRecorded, steps=12, field="deficit")
print("The farm's books close every month:", np.allclose(received, lost_on_the_way + used_up + drained_back))

In [ ]:
#| label: fig-demand
#| fig-cap: "The canal brings more than the farm needs in most months, so the farm uses what it needs, loses 15 % on the way, and sends the rest back. In September the canal brings 70 for a need of 67, only 60 reach the roots, and the farm is short."
#| echo: false
_ = plot_months(
    {"used up by the crops": used_up, "lost on the way": lost_on_the_way, "drained back": drained_back, "short": short},
    "A Demand: need, use, loss, return",
    kind="bar",
    stacked=True,
    reference=("the farm's need", np.array(FARM_NEED_MM3)),
)

Look at August first. The canal brought 142 for a need of 113; 15 % is lost before the roots, 121 arrive, and that
is enough. Now September:

In [ ]:
#| label: demand-table
#| echo: false
pd.DataFrame({"month": MONTHS, "need": FARM_NEED_MM3, "received": received.round(1), "reached the roots": (received - lost_on_the_way).round(1), "short": short.round(1)}).style.hide(axis="index")

The lesson is in the September row: the canal brings 70, the roots get 60, the need is 67. **Efficiency sits
between the gate and the need**, so a block upstream that delivers exactly the need leaves the customer short by the
loss. Keep that in mind for the reservoir's valve below.

# PassThrough: a gate of fixed width {#sec-passthrough}

A **PassThrough** is the simplest block with a number: a gate. Water up to its *capacity* goes through; the rest is
recorded as spilled. It models a turbine, a pump, a pipe, an intake: anything with a maximum. Without a capacity it
is just a measuring point.

In [ ]:
#| label: passthrough
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="river", inflow=TimeSeries(values=list(river))))
system.add_node(PassThrough(id="turbine", capacity=600.0))  # the turbine can take 600 Mm³ a month
system.add_node(Sink(id="sea"))
system.add_edge(Edge(id="river_to_turbine", source="river", target="turbine"))
system.add_edge(Edge(id="turbine_to_sea", source="turbine", target="sea"))
system.validate()
system.simulate(12)

turbine = system.nodes["turbine"]
through = monthly(turbine, WaterPassedThrough, steps=12)
past = monthly(turbine, WaterSpilled, steps=12)
print(f"The sea received {monthly(system.nodes['sea'], WaterReceived, steps=12).sum():,.0f} of the river's {river.sum():,.0f} Mm³; {past.sum():,.0f} spilled past the gate.")

In [ ]:
#| label: fig-passthrough
#| fig-cap: "June to August the river is wider than the gate. What does not fit is recorded as spilled and leaves the model."
#| echo: false
_ = plot_months({"through the gate": through, "spilled past it": past}, "A PassThrough: what fits goes through", kind="bar", stacked=True)

Here is the second kind of spill, and it is different from the bathtub's overflow: **water above a gate's capacity
leaves the books.** TaqSim records it and does not follow it further. In a real river the water that bypasses a
turbine is still in the river; if that matters for your question, the turbine belongs on a side branch of a
Splitter, or the capacity belongs on a Reach. Knowing which spill you are looking at is half of reading a result.

# All nine together {#sec-together}

Now the whole line: the river enters a canyon, reaches a headworks that sends a share into a canal and the rest to
a turbine on the river, the canal feeds the reservoir, the reservoir serves the farm, and both branches end in a
sink. Nine blocks of seven kinds, eight edges, five cards with a number in them.

In [ ]:
#| label: fig-network
#| fig-cap: "The demo system. Water moves left to right; the kind of each block is written under its name."
#| echo: false
_ = plot_demo_network()

`build_demo` wires it, with the five numbers collected in a `Knobs` record so that they are easy to change:

In [ ]:
#| label: build-card
#| echo: false
rule_card(Knobs)

In [ ]:
#| label: build-demo-card
#| echo: false
rule_card(build_demo)

`simulate_demo` runs three years and hands back the last one, after the reservoir has settled into its yearly
rhythm. With the knobs at their defaults:

In [ ]:
#| label: run
run = simulate_demo(Knobs())  # three years; the last one comes back, month by month, block by block

In [ ]:
#| label: fig-july
#| fig-cap: "July, the river's biggest month. Follow the numbers along the edges: 993 enters the canyon, the 771 of June comes out, 154 of it goes into the canal and 617 to the turbine, which can only take 600."
#| echo: false
_ = plot_demo_network(run, month=9)

In [ ]:
#| label: fig-where
#| fig-cap: "Where the water went, month by month. Most of it goes to the river end through the turbine; the canyon holds each month's water for a month (the light bars below zero are last month's water coming out); the reservoir fills in spring and drains in late summer."
#| echo: false
_ = plot_where_the_water_went(run)

## Every drop accounted for {#sec-balance}

Every block keeps its diary, so for every block we can check: what came in equals what went out, plus what was
used up, lost, left the model at a gate, or stayed in store. The table does that for the year.

In [ ]:
#| label: balance-table
#| echo: false
balance_table(run).round(1).style.hide(axis="index").format(precision=1)

`closes` is zero on every row. The reservoir's `out` includes its overflow, because the overflow went on to the
farm; the turbine's `left` is water that TaqSim stopped following. For the whole system in one picture:

In [ ]:
#| label: fig-balance
#| fig-cap: "The year adds up. What the river brought equals what reached the two sinks, what the crops used, what was lost, what left at the turbine's gate, and the change in storage (zero: the reservoir ended the year where it began)."
#| echo: false
_ = plot_balance(run)

In [ ]:
#| label: balance-numbers
#| echo: false
b = system_balance(run)
stored = 0.0 if abs(b["stored"]) < 0.5 else b["stored"]
print(f"River {b['river']:,.0f} = sinks {b['to_sinks']:,.0f} + used {b['consumed']:,.0f} + lost {b['lost']:,.0f} + left at the gate {b['left']:,.0f} + stored {stored:,.0f}")

This check is the habit to take from this notebook. A model whose blocks do not close their books has a wiring
mistake; a model that closes them can still be wrong, but at least every drop is somewhere you can point to.

## Turn a knob {#sec-knobs}

Two experiments. First, a tighter valve: the reservoir may release at most 20 Mm³ a month.

In [ ]:
#| label: tight-valve
tight = simulate_demo(Knobs(release_cap=20.0))
print(f"Valve at 20: the farm is short {tight.series['farm']['deficit'].sum():,.0f} Mm³ in the year, while the reservoir never drops below {tight.storage_level.min():,.0f} Mm³.")
print(f"Valve at 60: the farm is short {run.series['farm']['deficit'].sum():,.0f} Mm³.")

In [ ]:
#| label: fig-tight
#| fig-cap: "A full reservoir and a thirsty farm: with the valve at 20 the farm is short in May and June while the tub holds over 600 Mm³. The shortfall is the rule's doing, not the river's."
#| echo: false
_ = plot_months(
    {"short, valve at 20": tight.series["farm"]["deficit"], "short, valve at 60": run.series["farm"]["deficit"]},
    "The same river, two valves",
    kind="bar",
    reference=("the farm's need", np.array(FARM_NEED_MM3)),
)

Second, a narrower gate at the turbine, 300 instead of 600:

In [ ]:
#| label: narrow-gate
narrow = simulate_demo(Knobs(turbine_capacity=300.0))
print(f"Gate at 300: {narrow.series['turbine']['spilled'].sum():,.0f} Mm³ spill past the turbine and leave the model, against {run.series['turbine']['spilled'].sum():,.0f} at 600.")
print(f"The river end receives {narrow.flows['turbine_to_river_end'].sum():,.0f} instead of {run.flows['turbine_to_river_end'].sum():,.0f} Mm³.")

Why is the tight valve's shortfall not larger? Look at the months: the farm is short in May and June, when the
valve is the only way out of the tub. From July on the tub is full, the melt overflows, and the overflow reaches
the farm whatever the valve says. Two lessons from this notebook in one experiment: a rule can cause a shortfall,
and a full reservoir's overflow travels on.

Change any of the five numbers in `Knobs(...)`, run the cells above again, and read the diaries. The game page
of this step does the same with sliders, for every combination, precomputed.

## What to notice {#sec-notice}

1. **The canyon delays the river by a month.** The July peak reaches the headworks in August. A reach with a lag
   changes *when* water arrives, not how much; a reach with a loss changes how much.
2. **Two spills, two meanings.** A full reservoir's overflow goes on down the edge. A gate's spill leaves the books.
   The diary entry has the same name, `WaterSpilled`; the block it sits in tells you which it is.
3. **A shortfall can be a rule's doing.** With the valve at 20 the farm is short in May and June while the
   reservoir holds over 600 Mm³. Before blaming the river, read the rule.
4. **Efficiency sits between the gate and the need.** Even with a wide valve the farm is a little short, because the
   valve was told the need at the gate and 15 % is lost after it. A rule that knows about the loss would release
   the need divided by the efficiency.

# Try it yourself {#sec-try}

1. **A seasonal split.** Write a split rule `SummerShare` that sends 40 % into the canal from April to September
   (`t.index % 12` is 6 to 11) and 10 % in the other months. Put it on the headworks of the demo, run, and compare
   the reservoir's level path with the default.
2. **A wiser valve.** Change `ReleaseTheNeed` so that it releases the need divided by the farm's efficiency. Does the
   farm's shortfall go to zero in every month? Which months still fail, and why?
3. **Evaporation that follows the season.** Make `Evaporation` take 1 % in winter and 5 % in summer. How much more
   water reaches the farm over the year?
4. **A second customer.** Add a city as a `Demand` of 10 Mm³ a month between the reservoir and the farm (the city
   returns 80 % of its water). Who is short first in a dry year, and what does the order of the blocks have to do
   with it?

**With Gemini in Colab.** Click the blue Gemini button and paste one of these; the notebook's own code is the only
TaqSim reference it has, so point it at the right cell:

> Using the `CanalShare` class in this notebook as a model, write a split rule that sends 40 % into the canal in
> months 6 to 11 of each year and 10 % otherwise. Then build the demo system with it and print the reservoir's
> minimum level.

> Change `ReleaseTheNeed` so that it releases `need / 0.85` instead of `need`, keeping the cap. Run `simulate_demo`
> with the new rule and print the farm's yearly deficit before and after.

# Where to go next {#sec-next}

The starter notebook of this course has the one-page cheat sheet of every block and rule in TaqSim v0.1.4, written
for an AI helper to read. The reservoir-and-farm coursebook builds a four-block system and lets the optimizer turn
the knobs. The simulation coursebook is the real thing: 61 blocks, six years, day by day, built from exactly these
seven kinds. And TaqSim ships its own documentation inside the package: `from taqsim import get_docs_path`.

# Sources {#sec-sources .unnumbered}

- The river: Zarafshan at Ravatkhoja, daily record 2010–2023, median month (`data/ZRB_baseline/inflow/`).
- The farm: irrigation need of the Kattakurgan district, monthly, from the Zarafshan model of this course.
- The reservoir: Kattakurgan, capacity 667 Mm³, dead storage 66.7 Mm³, 170 Mm³ on the first of October (median
  2000–2022, see the sixteen-blocks game).
- TaqSim v0.1.4, hydrosolutions GmbH, https://github.com/hydrosolutions/taqsim, and its documentation on nodes,
  events and strategies.